# Extracción de priors desde un light curve (TIC)

Versión optimizada del pipeline original. Cambios principales:

- **Una sola función de entrada** (`run_pipeline`): carga, corre BLS + Lomb-Scargle, mide la profundidad del eclipse, grafica y guarda el prior en el catálogo.
- **`depth` se calcula dentro del pipeline principal** (antes solo existía en la celda de la figura de publicación).
- **`load_lightcurve` más robusta**: detecta separador, valida columnas, quita duplicados/NaN, ordena por tiempo.
- **`save_priors` unifica** a las antiguas `save_priors`/`append_priors`, guarda todos los valores y no se cuelga en modo no interactivo.
- **La clave de cada fila del catálogo es (TIC, Sector)**, no solo TIC: puedes guardar el mismo TIC varias veces, una por cada sector observado, sin que se pisen entre sí. Repetir el mismo TIC+sector sí actualiza esa fila puntual.
- **TIC y sector se detectan automáticamente** del nombre del archivo (patrón `tic 12345_S_67_LC.txt`) si no los pasas a mano.

**Lo único que tienes que editar es la última celda**, donde pones la ruta a tu archivo.


In [ ]:
# ============================================================
# EXTRACCION DE PRIORS DESDE UN LIGHT CURVE (TIC)
# ============================================================
# Dado un archivo de light curve local, este script:
#   1) lo carga,
#   2) lo aplana (wotan) para separar señal orbital / rotacional,
#   3) busca el período orbital con BLS,
#   4) decide si el período real es P o 2P (binaria eclipsante),
#   5) mide la profundidad del eclipse,
#   6) busca el período de rotación con Lomb-Scargle,
#   7) guarda todo en un catálogo priors.csv
# ============================================================

import os
import re
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.timeseries import BoxLeastSquares, LombScargle
from wotan import flatten

# ------------------------------------------------------------
# CONFIGURACION POR DEFECTO (se puede sobreescribir por llamada)
# ------------------------------------------------------------


## Configuración por defecto
Puedes sobreescribir cualquiera de estos valores pasándolos como argumento a `run_pipeline(...)` o `analyze_system(...)`, sin tocar esta celda.

In [ ]:
DEFAULTS = dict(
    time_column="#Time",
    flux_column="Flux_CBV_SS",

    window_length=2.0,      # días, ventana de wotan
    width_factor=3.0,       # ancho de la máscara de eclipse (x duración)

    min_period_bls=0.3,
    max_period_bls=15.0,
    n_periods_bls=2000,

    min_period_ls=0.2,
    max_period_ls=15.0,
    n_freq_ls=2000,

    eclipse_half_width=0.03,   # en fase, para medir profundidad
    out_of_eclipse_lo=0.15,
    out_of_eclipse_hi=0.35,
)


## 1) Carga del archivo

In [ ]:
# ============================================================
# 1) CARGA DEL LIGHT CURVE
# ============================================================

def load_lightcurve(filename, time_column="#Time", flux_column="Flux_CBV_SS",
                     sep=None):
    """
    Carga un light curve desde disco. Detecta el separador si no se indica
    (coma para .csv, espacios/tabs para todo lo demás) y valida que las
    columnas pedidas existan, con un error claro si no.
    """
    if not os.path.exists(filename):
        raise FileNotFoundError(f"No se encontró el archivo: {filename}")

    if sep is None:
        sep = "," if filename.lower().endswith(".csv") else r"\s+"

    df = pd.read_csv(filename, sep=sep, engine="python")

    # normaliza nombres de columnas por si vienen con espacios extra
    df.columns = [c.strip() for c in df.columns]

    if time_column not in df.columns or flux_column not in df.columns:
        raise KeyError(
            f"Columnas '{time_column}' / '{flux_column}' no encontradas.\n"
            f"Columnas disponibles: {list(df.columns)}"
        )

    time = df[time_column].values.astype(float)
    flux = df[flux_column].values.astype(float)

    mask = np.isfinite(time) & np.isfinite(flux)
    time, flux = time[mask], flux[mask]

    # ordena por tiempo (por si el archivo no viene ordenado) y quita
    # timestamps duplicados, que rompen BLS/Lomb-Scargle
    order = np.argsort(time)
    time, flux = time[order], flux[order]
    _, unique_idx = np.unique(time, return_index=True)
    time, flux = time[unique_idx], flux[unique_idx]

    if len(time) < 50:
        raise ValueError(
            f"Muy pocos puntos válidos ({len(time)}) tras limpiar el archivo."
        )

    return time, flux


def parse_tic_sector_from_filename(filename):
    """
    Intenta extraer TIC y sector del nombre del archivo, siguiendo la
    convención usada en los archivos originales, p.ej.:
    'tic 11300044_S_98_LC.txt' -> tic=11300044, sector=98
    Devuelve (None, None) si no logra reconocer el patrón.
    """
    name = os.path.basename(filename)
    m = re.search(r"tic\s*0*(\d+).*?_S_0*(\d+)", name, re.IGNORECASE)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None, None


## 2) Búsqueda del período orbital (BLS)

In [ ]:
# ============================================================
# 2) BLS (período orbital)
# ============================================================

def run_bls(time, flux, min_period, max_period, n_periods):
    periods = np.linspace(min_period, max_period, n_periods)
    durations = np.linspace(0.005, 0.9 * min_period, 100)

    bls = BoxLeastSquares(time, flux)
    result = bls.power(periods, durations)

    idx = np.argmax(result.power)

    return {
        "P_bls": result.period[idx],
        "t0": result.transit_time[idx],
        "duration": result.duration[idx],
        "period_grid": result.period,
        "power_grid": result.power,
    }


def determine_orbital_period(time, flux, t0, P_bls):
    """Decide si el período real es P_bls o 2*P_bls comparando
    la profundidad del mínimo primario vs secundario."""
    P1, P2 = P_bls, 2 * P_bls

    phase = ((time - t0) / P2) % 1

    primary = np.nanmedian(flux[np.abs(phase - 0.0) < 0.05])
    secondary = np.nanmedian(flux[np.abs(phase - 0.5) < 0.05])

    if abs(primary - secondary) > 0.005:
        return P2
    return P1


def build_mask(time, period, t0, duration, width_factor=3.0):
    phase = ((time - t0) / period) % 1
    width = width_factor * duration / period

    primary = (phase < width) | (phase > 1 - width)
    secondary = np.abs(phase - 0.5) < width

    return primary | secondary


## 3) Profundidad del eclipse

In [ ]:
# ============================================================
# 3) PROFUNDIDAD DEL ECLIPSE
# ============================================================

def compute_depth(time, flux, t0, period, half_width=0.03,
                   out_lo=0.15, out_hi=0.35):
    phase = ((time - t0) % period) / period

    in_eclipse = np.abs(phase - 0.5) < half_width
    out_eclipse = (phase > out_lo) & (phase < out_hi)

    if np.sum(in_eclipse) < 5 or np.sum(out_eclipse) < 5:
        return None, None

    baseline = np.nanmedian(flux[out_eclipse])
    eclipse = np.nanmedian(flux[in_eclipse])
    depth = baseline - eclipse

    depth_err = np.nanstd(flux[in_eclipse]) / np.sqrt(np.sum(in_eclipse))

    return float(depth), float(depth_err)


## 4) Período de rotación (Lomb-Scargle)

In [ ]:
# ============================================================
# 4) LOMB-SCARGLE (período de rotación)
# ============================================================

def run_ls(time, flux, min_period, max_period, n_freq):
    freq = np.linspace(1 / max_period, 1 / min_period, n_freq)

    ls = LombScargle(time, flux)
    power = ls.power(freq)

    idx = np.argmax(power)

    return {
        "P_rot": 1 / freq[idx],
        "frequency": freq,
        "power": power,
    }


## 5) Pipeline principal (`analyze_system`)

In [ ]:
# ============================================================
# 5) PIPELINE PRINCIPAL
# ============================================================

def analyze_system(filename, **kwargs):
    """
    Corre el pipeline completo sobre un light curve y devuelve un dict
    con todo lo necesario (curvas, períodos, profundidad, etc).
    Cualquier parámetro en DEFAULTS puede sobreescribirse por kwargs,
    p.ej. analyze_system(archivo, window_length=0.9, min_period_bls=0.5)
    """
    cfg = {**DEFAULTS, **kwargs}

    time, flux = load_lightcurve(
        filename,
        time_column=cfg["time_column"],
        flux_column=cfg["flux_column"],
    )

    # -------- detrend --------
    orbital_flux, rotational_trend = flatten(
        time, flux,
        method="biweight",
        window_length=cfg["window_length"],
        break_tolerance=0.5,
        return_trend=True,
    )

    # -------- BLS --------
    orbital = run_bls(
        time, orbital_flux,
        cfg["min_period_bls"], cfg["max_period_bls"], cfg["n_periods_bls"],
    )

    P_orb = determine_orbital_period(
        time, orbital_flux, orbital["t0"], orbital["P_bls"]
    )

    # -------- profundidad --------
    depth, depth_err = compute_depth(
        time, orbital_flux, orbital["t0"], P_orb,
        half_width=cfg["eclipse_half_width"],
        out_lo=cfg["out_of_eclipse_lo"],
        out_hi=cfg["out_of_eclipse_hi"],
    )

    # -------- máscara de eclipses para la rotación --------
    mask = build_mask(
        time, P_orb, orbital["t0"], orbital["duration"], cfg["width_factor"]
    )

    time_rot, flux_rot = time[~mask], flux[~mask]

    if len(time_rot) < 20:
        # si casi todo cae en eclipse, usa la curva completa
        time_rot, flux_rot = time, flux

    rotation = run_ls(
        time_rot, flux_rot,
        cfg["min_period_ls"], cfg["max_period_ls"], cfg["n_freq_ls"],
    )

    tic, sector = parse_tic_sector_from_filename(filename)

    return {
        "filename": filename,
        "tic": tic,
        "sector": sector,

        "time": time,
        "flux": flux,

        "orbital_flux": orbital_flux,
        "rotational_flux": rotational_trend,

        "time_rot": time_rot,
        "flux_rot": flux_rot,
        "mask": mask,

        "P_bls": orbital["P_bls"],
        "P_orb": P_orb,
        "P_rot": rotation["P_rot"],
        "t0": orbital["t0"],
        "duration": orbital["duration"],
        "depth": depth,
        "depth_err": depth_err,

        "orbital": orbital,
        "rotation": rotation,
        "config": cfg,
    }


## 6) Figura de diagnóstico

In [ ]:
# ============================================================
# 6) FIGURA DE DIAGNOSTICO
# ============================================================

def plot_diagnostics(results, savepath=None):
    time, flux = results["time"], results["flux"]
    orbital_flux = results["orbital_flux"]
    rotational_flux = results["rotational_flux"]

    P_orb, P_rot, t0 = results["P_orb"], results["P_rot"], results["t0"]

    phase_orb = ((time - t0) / P_orb) % 1
    phase_rot = ((time - np.nanmin(time)) / P_rot) % 1

    idx_orb = np.argsort(phase_orb)
    idx_rot = np.argsort(phase_rot)

    fig, ax = plt.subplots(2, 2, figsize=(14, 10))

    ax[0, 0].plot(time, flux, ".", ms=1, alpha=0.25, label="Original")
    ax[0, 0].plot(time, rotational_flux, lw=2, label="Wotan trend")
    ax[0, 0].set_title("(a) Detrending")
    ax[0, 0].set_xlabel("Time (days)"); ax[0, 0].set_ylabel("Flux")
    ax[0, 0].legend()

    ax[0, 1].plot(phase_rot[idx_rot], rotational_flux[idx_rot], ".", ms=2, alpha=0.4)
    ax[0, 1].set_title(f"(b) Fase rotacional\nP_rot = {P_rot:.5f} d")
    ax[0, 1].set_xlabel("Phase"); ax[0, 1].set_ylabel("Flux")

    ax[1, 0].plot(time, orbital_flux, ".", ms=2, alpha=0.5)
    ax[1, 0].set_title("(c) Curva aplanada")
    ax[1, 0].set_xlabel("Time (days)"); ax[1, 0].set_ylabel("Normalized flux")

    ax[1, 1].plot(phase_orb[idx_orb], orbital_flux[idx_orb], ".", ms=2, alpha=0.5)
    title_d = f"(d) Fase\nP_orb = {P_orb:.5f} d"
    if results["depth"] is not None:
        title_d += f"  |  depth = {results['depth']*100:.3f}%"
    ax[1, 1].set_title(title_d)
    ax[1, 1].set_xlabel("Phase"); ax[1, 1].set_ylabel("Normalized flux")

    fig.suptitle(
        f"TIC {results['tic']}  Sector {results['sector']}   |   "
        f"P_orb = {P_orb:.5f} d   |   P_rot = {P_rot:.5f} d",
        fontsize=13,
    )
    plt.tight_layout()

    if savepath:
        plt.savefig(savepath, dpi=200, bbox_inches="tight")
    plt.show()


## 7) Guardar priors en el catálogo (`priors.csv`)
La fila se identifica por (TIC, Sector): el mismo TIC con distinto sector se agrega como fila nueva; repetir TIC+sector actualiza esa fila.

In [ ]:
# ============================================================
# 7) GUARDAR PRIORS EN EL CATALOGO
# ============================================================

def save_priors(results, tic=None, sector=None, quality="A", notes="",
                 filename="priors.csv", overwrite=None):
    """
    Guarda/actualiza una fila en el catálogo de priors con TODOS los
    valores relevantes (P_bls, P_orb, P_rot, t0, duration, depth, ...).
    Reemplaza a las antiguas save_priors()/append_priors() por una sola
    función consistente.

    La clave que identifica una fila es (TIC, Sector), no solo TIC: así
    puedes guardar el mismo TIC varias veces, una por cada sector, sin que
    se pisen entre sí. Dos filas solo se consideran "la misma" si además
    de compartir TIC comparten sector (incluyendo el caso sector=None).

    overwrite:
        True  -> sobreescribe sin preguntar si ya existe (mismo TIC+sector)
        False -> nunca sobreescribe, deja la fila existente
        None  -> pregunta por consola si es posible; si no hay consola
                 interactiva (script/notebook batch), sobreescribe y avisa
    """
    tic = tic if tic is not None else results.get("tic")
    sector = sector if sector is not None else results.get("sector")

    if tic is None:
        raise ValueError("No se pudo determinar el TIC: pásalo explícitamente.")

    row = {
        "TIC": tic,
        "Sector": sector,
        "P_bls": results["P_bls"],
        "P_orb": results["P_orb"],
        "P_rot": results["P_rot"],
        "t0": results["t0"],
        "Duration": results["duration"],
        "Depth": results["depth"],
        "Depth_err": results["depth_err"],
        "Quality": quality,
        "Notes": notes,
    }

    if not os.path.exists(filename):
        pd.DataFrame([row]).to_csv(filename, index=False)
        print(f"Catálogo creado: {filename}")
        return

    df = pd.read_csv(filename)

    same_tic = df["TIC"] == tic
    # compara sector con cuidado de NaN==NaN (que en pandas da False)
    if sector is None:
        same_sector = df["Sector"].isna()
    else:
        same_sector = df["Sector"] == sector
    mask = same_tic & same_sector

    label = f"TIC {tic} / sector {sector}"

    if mask.any():
        do_overwrite = overwrite
        if do_overwrite is None:
            if sys.stdin.isatty():
                ans = input(f"{label} ya existe. ¿Sobreescribir? (y/n): ")
                do_overwrite = ans.strip().lower() == "y"
            else:
                print(f"{label} ya existe; sobreescribiendo (modo no interactivo).")
                do_overwrite = True

        if do_overwrite:
            # se borra la fila vieja y se agrega la nueva en vez de asignar
            # celda por celda, para evitar conflictos de dtype entre pandas
            # y valores mixtos (texto/números/NaN) en la misma columna
            df = df.loc[~mask]
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
            print(f"{label}: entrada actualizada.")
        else:
            print(f"{label}: entrada NO modificada.")
            return
    else:
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        print(f"{label}: entrada nueva agregada.")

    df = df.sort_values(["TIC", "Sector"], na_position="last")
    df.to_csv(filename, index=False)
    print(f"Guardado en {filename}")


## 8) Pipeline de un solo paso (`run_pipeline`)
Esta es la función que realmente vas a usar.

In [ ]:
# ============================================================
# 8) PIPELINE DE UN SOLO PASO
# ============================================================

def run_pipeline(filename, tic=None, sector=None, quality="A", notes="",
                  show_plot=True, save_csv=True, catalog="priors.csv",
                  **kwargs):
    """
    Una sola llamada: carga -> analiza -> imprime resumen -> (opcional)
    grafica -> (opcional) guarda en el catálogo.
    """
    results = analyze_system(filename, **kwargs)

    if tic is not None:
        results["tic"] = tic
    if sector is not None:
        results["sector"] = sector

    print(f"Archivo:   {filename}")
    print(f"TIC:       {results['tic']}   Sector: {results['sector']}")
    print(f"P_BLS  =   {results['P_bls']:.6f} d")
    print(f"P_orb  =   {results['P_orb']:.6f} d")
    print(f"P_rot  =   {results['P_rot']:.6f} d")
    print(f"t0     =   {results['t0']:.6f}")
    print(f"duration = {results['duration']:.6f} d")
    if results["depth"] is not None:
        print(f"depth  =   {results['depth']*100:.4f} % ± {results['depth_err']*100:.4f} %")
    else:
        print("depth  =   no se pudo medir (revisa la máscara / cobertura)")

    if show_plot:
        plot_diagnostics(results)

    if save_csv:
        save_priors(results, tic=results["tic"], sector=results["sector"],
                    quality=quality, notes=notes, filename=catalog)

    return results


## Uso

Edita `FILE` con la ruta a tu light curve y corre la celda. Si tu archivo no sigue la convención `tic <TIC>_S_<sector>_LC.txt`, o tus columnas no se llaman `#Time`/`Flux_CBV_SS`, pasa `tic=`, `sector=`, `time_column=` y/o `flux_column=` explícitamente. Para procesar varios sectores del mismo objeto, corre esta celda una vez por cada archivo/sector: cada corrida se agrega como fila nueva en `priors.csv` sin pisar las anteriores.


In [ ]:
file = "/home/luciam/Documents/starry/Orion_yceb/dat_orion_sergio/sample_3/831359(1)/Light_Curve_Files/tic 11300044_S_98_LC.txt"  # <-- cambia esto por tu archivo

results = run_pipeline(
    file,
    # tic=12345,            # descomenta si el nombre del archivo no trae el TIC
    # sector=10,            # descomenta si el nombre del archivo no trae el sector
    # time_column="#Time",
    # flux_column="Flux_CBV_SS",
    window_length=0.75,      # ventana de wotan, ajusta según tu curva
    show_plot=True,
    save_csv=True,
    catalog="priors.csv",
    quality="A",
    notes="",
)
